# 06 — Model Selection, Stability, and Explainability

This notebook focuses on judgement: how stable is the structure we discovered?


In [1]:
import sys
from pathlib import Path

from sklearn.cluster import KMeans

sys.path.append(str(Path.cwd() / "src"))

from unsup_lab.data import make_customer_segmentation_data
from unsup_lab.evaluation import evaluate_k_range
from unsup_lab.preprocessing import scale_features
from unsup_lab.reporting import cluster_profile
from unsup_lab.stability import (
    bootstrap_cluster_stability,
    outlier_sensitivity,
    pairwise_adjusted_mutual_information,
    repeated_run_labels,
    scaling_sensitivity,
    stability_report,
)

In [2]:
dataset = make_customer_segmentation_data(n_customers=1_500, random_state=123)
features = dataset.features
scaled = scale_features(features, method="standard")


# A factory maps a random seed to a fresh estimator, so the stability
# tools can re-fit the same configuration under different randomness.
def kmeans_factory(seed: int) -> KMeans:
    return KMeans(n_clusters=5, n_init=10, random_state=seed)

## Internal metrics across k

In [3]:
metrics = evaluate_k_range(
    scaled.to_numpy(),
    estimator_factory=lambda k: KMeans(n_clusters=k, n_init=20, random_state=123),
    k_values=list(range(2, 11)),
)

metrics.round(3)

,k,n_clusters,silhouette,davies_bouldin,calinski_harabasz
0,2,2,0.405,1.147,1000.555
1,3,3,0.448,0.954,1065.182
2,4,4,0.533,0.719,1454.594
3,5,5,0.623,0.552,2743.698
4,6,6,0.510,1.073,2310.453
5,7,7,0.402,1.447,2002.911
6,8,8,0.321,1.737,1773.639
7,9,9,0.230,1.993,1607.921
8,10,10,0.226,1.937,1482.757


## Seed stability

In [4]:
# Re-fit KMeans under 20 seeds and measure how much the partitions agree.
seed_runs = repeated_run_labels(scaled.to_numpy(), kmeans_factory, n_runs=20, random_state=0)
seed_stability = pairwise_adjusted_mutual_information(seed_runs)
seed_stability

StabilitySummary(mean_ami=1.0, std_ami=1.0685374877934257e-16, n_pairs=190, mean_n_clusters=5.0)

## Bootstrap stability

Seed agreement only varies the initialisation. Bootstrap stability also varies *which customers are present*, which is a stronger test: if the segments survive resampling, they are unlikely to be an artefact of a few rows.

In [5]:
bootstrap = bootstrap_cluster_stability(
    scaled.to_numpy(), kmeans_factory, n_bootstrap=20, sample_fraction=0.8, random_state=0
)
bootstrap

StabilitySummary(mean_ami=1.0, std_ami=7.381983537828248e-17, n_pairs=190, mean_n_clusters=5.0)

## Sensitivity to scaling and outliers

Two more failure modes worth checking before trusting a segmentation: does the partition depend on the scaling choice, and does a handful of injected outliers reshuffle the assignments?

In [6]:
scaling = scaling_sensitivity(features, kmeans_factory, random_state=0)
scaling.round(3)

,scaling,n_clusters,silhouette,ami_vs_reference
0,none,5,0.492,0.793
1,standard,5,0.623,1.000
2,robust,5,0.639,1.000


In [7]:
outliers = outlier_sensitivity(
    scaled.to_numpy(),
    kmeans_factory,
    contamination_levels=(0.0, 0.02, 0.05, 0.1),
    random_state=0,
)
outliers.round(3)

,contamination,n_clusters,silhouette,ami_vs_baseline
0,0.00,5,0.623,1.000
1,0.02,5,0.564,1.000
2,0.05,5,0.444,0.907
3,0.10,5,0.402,0.890


## Explain the selected clustering

In [8]:
selected = kmeans_factory(123).fit_predict(scaled.to_numpy())
profile = cluster_profile(features, selected)

profile.round(2)

,recency_days,purchase_frequency,avg_order_value,discount_ratio,email_engagement,product_diversity,cluster_size
cluster,,,,,,,
0,9.76,36.58,31.23,0.33,0.73,0.66,307
1,11.99,3.63,38.19,0.28,0.23,0.16,322
2,18.29,24.71,187.95,0.13,0.82,0.73,251
3,33.81,13.90,71.30,0.72,0.61,0.59,379
4,109.68,7.35,115.76,0.20,0.38,0.29,241


## Automated stability report

`stability_report` bundles every diagnostic above into a single Markdown summary that can be saved as an artefact or pasted into a review.

In [9]:
from IPython.display import Markdown, display

report = stability_report(scaled, kmeans_factory, n_runs=10, n_bootstrap=10, random_state=0)
display(Markdown(report))

# Clustering stability report

## Seed agreement (repeated runs)
- Mean pairwise AMI: **1.000** (std 0.000, 45 pairs)
- Mean clusters per run: 5.00

## Bootstrap stability
- Mean pairwise AMI on shared points: **1.000** (std 0.000, 45 pairs)
- Mean clusters per subsample: 5.00

## Scaling sensitivity
| scaling | n_clusters | silhouette | ami_vs_reference |
| --- | --- | --- | --- |
| none | 5 | 0.623 | 1.000 |
| standard | 5 | 0.623 | 1.000 |
| robust | 5 | 0.639 | 1.000 |

## Outlier sensitivity
| contamination | n_clusters | silhouette | ami_vs_baseline |
| --- | --- | --- | --- |
| 0.000 | 5 | 0.623 | 1.000 |
| 0.020 | 5 | 0.564 | 1.000 |
| 0.050 | 5 | 0.444 | 0.907 |
| 0.100 | 5 | 0.402 | 0.890 |


## Interpretation

A good unsupervised learning workflow should ask:

- Does the result change under another random seed?
- Does it change under another scaling method?
- Does it change when outliers are removed?
- Are the clusters actionable?
- Are the clusters stable enough to support decisions?

These questions are often more important than the choice of algorithm.
